# 01.7 — ADME MMP Analysis

Recreates the Matched Molecular Pair (MMP) analysis from Fang et al. (2023), §5.4/5.5, on the ADME dataset the paper released publicly (their MMP analysis itself used >25,000 confidential in-house compounds, never shared). Uses [`mmpdb`](https://github.com/rdkit/mmpdb) via `src/mmp/` (see [`src/mmp/CLAUDE.md`](../src/mmp/CLAUDE.md)) to fragment + index the dataset and extract statistically significant transformation rules per endpoint.

**Data source**: molecules and labels come from `df_sdf` (`data/processed/section4_df_sdf.pkl`), the same SDF-standardized, ChEMBL-augmented dataset built in `01.5_adme_biogen_public_recreation.ipynb` §2.4 — not the raw CSV. Two reasons: (1) the CSV and SDF standardize molecules slightly differently, which would shift fragmentation/pairing versus the paper's own approach; (2) the SDF recovers ChEMBL-augmented PPB_H/PPB_R data (1795/876 non-missing vs ~170 in the CSV), so all 6 endpoints — HLM, MDR1, SOL, RLM, PPB_H, PPB_R — are covered here, not just the 4 used for ML modelling elsewhere in this project.

**Method (matches the paper, confirmed against mmpdb source):**
- Fragmentation & indexing use mmpdb's own defaults — the paper's quoted cut-SMARTS/heavy-atom/rotatable-bond parameters are literally mmpdb's built-in defaults, not a custom rule set. Pinned explicitly in `src/mmp/mmp.py` so a future mmpdb version can't silently drift.
- "Representative rules" = per-rule statistics with ≥5 matched pairs, paired-t-test p<0.05, at the most specific environment radius ≤3 — mmpdb always computes radius 0–5 at index time; the paper's "max radius 3" is a query-time selection, reimplemented in `significant_rules()`.

**Two known deviations from the paper (unresolved, stated here rather than left implicit):**
- **mmpdb version**: the paper used mmpdb 2.2-dev1; this project has `mmpdb==2.1` installed (`uv add mmpdb` resolves to 2.1, the newest version compatible with this project's `rdkit==2023.9.5` pin — mmpdb≥3.x requires `rdkit>=2024.3`, confirmed to conflict directly). Fragmentation/indexing behavior across that version gap isn't verified identical.
- **"Minimum heavy atoms per constant fragment = 0"**: stated in the paper's appendix. mmpdb 2.1's `fragment` CLI has no corresponding flag at all. However, 0 is a null constraint ("no minimum" = no filtering), which is exactly what mmpdb 2.1 already does by omitting the flag — confirmed via mmpdb 3.1.4 (which does have `--min-heavies-per-const-frag`) that setting it to 0 produces byte-identical downstream rule statistics to 2.1's behavior. So this is a documentation gap, not a results gap.

**Scope**: straight recreation only (single run on the full dataset). 

## 1 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import time
from pathlib import Path

import joblib
import pandas as pd
import matplotlib.pyplot as plt

from src.mmp import write_smi_file, write_properties_file, run_fragment, run_index, significant_rules

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
MMP_DIR = DATA_PROC / 'mmp'
MMP_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINTS = {
    'HLM':   'LOG HLM_CLint (mL/min/kg)',
    'MDR1':  'LOG MDR1-MDCK ER (B-A/A-B)',
    'SOL':   'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'RLM':   'LOG RLM_CLint (mL/min/kg)',
    'PPB_H': 'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
    'PPB_R': 'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
}

## 2 — Load data and write mmpdb input files

Loads `df_sdf` (standardized molecules, keyed by canonical SMILES, no separate compound-ID column — one is generated below). `mmpdb`'s property-file format requires short, whitespace-free property names, so we rename the endpoint columns to their short forms before writing.

In [ ]:
df = joblib.load(DATA_PROC / 'section4_df_sdf.pkl')
df = df.rename(columns={long: short for short, long in ENDPOINTS.items()})
df.insert(0, 'compound_id', [f'mol{i}' for i in range(len(df))])
print(df.shape)
df[['compound_id', 'can_smi'] + list(ENDPOINTS.keys())].head()

In [ ]:
smi_path = MMP_DIR / 'adme.smi'
props_path = MMP_DIR / 'adme_props.csv'

write_smi_file(df, smiles_col='can_smi', id_col='compound_id', path=smi_path)
write_properties_file(df, id_col='compound_id', property_cols=list(ENDPOINTS.keys()), path=props_path)

## 3 — Fragment + index

Builds the MMP SQLite database. Takes well under a minute on this dataset size.

In [ ]:
fragments_path = MMP_DIR / 'adme.fragments'
db_path = MMP_DIR / 'adme.mmpdb'

t0 = time.time()
run_fragment(smi_path, fragments_path, num_jobs=8)
print(f'fragment: {time.time() - t0:.1f}s')

t0 = time.time()
run_index(fragments_path, props_path, db_path)
print(f'index: {time.time() - t0:.1f}s')

## 4 — Database scale vs the paper

The paper built its database from >25,000 internal compounds and reported >1.2M rules. Checking how our public set compares before filtering for significance.

In [ ]:
con = sqlite3.connect(db_path)
n_compounds_indexed = con.execute('select count(*) from compound').fetchone()[0]
n_pairs = con.execute('select count(*) from pair').fetchone()[0]
n_candidate_rules = con.execute('select count(*) from rule').fetchone()[0]
con.close()

print(f'compounds indexed (>= 1 non-missing property): {n_compounds_indexed} / {len(df)}')
print(f'matched pairs: {n_pairs}')
print(f'candidate rules (pre-significance-filter): {n_candidate_rules}')
print('paper: >25,000 compounds -> >1.2M rules')

### 4.1 — Why the rule/pair counts above don't mean that many *usable* rules per endpoint

**What a "rule" and a "pair" actually are**: mmpdb first asks a purely structural question about the whole compound set — "which molecules are near-neighbours (differ by one small, localized change), and what exactly changed?" A **pair** is one such neighbour relationship (e.g. molecule #12 and molecule #47, which differ by swapping a fluorine for a hydrogen). A **rule** is the *type* of change involved (e.g. "F → H"), pooled across every pair that makes that same swap anywhere in the dataset. Neither `rule` nor `pair` (§4, above) knows or cares whether either molecule was ever tested in the HLM/MDR1/SOL/RLM/PPB_H/PPB_R assays — they're computed from structure alone.

**Where the assay endpoint comes in**: separately, each compound has (or is missing) a measured value for each of the 6 endpoints — coverage differs endpoint by endpoint, since not every compound was run through every assay. A rule like "F → H" only becomes *usable* for, say, HLM once we restrict to the subset of its pairs where **both** molecules in the pair have a non-missing HLM value. If "F → H" occurs in 40 pairs overall but only 3 of those pairs have HLM measured on both sides, then for HLM that rule effectively has just 3 data points — likely far short of the paper's 5-pair minimum, even though the same rule might clear that bar easily for an endpoint with better coverage.

Walking that funnel by hand for one endpoint (HLM) first, then generalizing to all six below.

In [ ]:
def count_rules_with_stats(con, property_id, min_pairs=0, max_p_value=None):
    """Count distinct rules with a rule_environment_statistics row (radius<=3, count>=min_pairs) for this property."""
    query = '''select count(distinct r.id)
               from rule r
               join rule_environment re on re.rule_id = r.id
               join rule_environment_statistics res on res.rule_environment_id = re.id
               where res.property_name_id = ? and re.radius <= 3 and res.count >= ?'''
    params = [property_id, min_pairs]
    if max_p_value is not None:
        query += ' and res.p_value < ?'
        params.append(max_p_value)
    return con.execute(query, params).fetchone()[0]

In [ ]:
con = sqlite3.connect(db_path)
hlm_property_id = con.execute('select id from property_name where name = ?', ('HLM',)).fetchone()[0]

n_hlm_raw = df['HLM'].notna().sum()
print(f'raw dataset: {n_hlm_raw} / {len(df)} compounds have an HLM value')

n_hlm_indexed_with_property = con.execute(
    'select count(distinct compound_id) from compound_property where property_name_id = ?',
    (hlm_property_id,),
).fetchone()[0]
print(f'of the {n_compounds_indexed} indexed compounds (those with >=1 structural match), '
      f'{n_hlm_indexed_with_property} still have an HLM value')

n_hlm_any_stat = count_rules_with_stats(con, hlm_property_id)
print(f'of the {n_candidate_rules} candidate rules, only {n_hlm_any_stat} have >=1 matched pair '
      f'where both molecules have HLM data')

n_hlm_min_pairs = count_rules_with_stats(con, hlm_property_id, min_pairs=5)
print(f"of those, only {n_hlm_min_pairs} clear the paper's >=5-matched-pair minimum")

n_hlm_significant = count_rules_with_stats(con, hlm_property_id, min_pairs=5, max_p_value=0.05)
print(f'and {n_hlm_significant} of those also pass p<0.05 -- this is the HLM row shown in §5 below')

con.close()

### 4.2 — Same funnel, all six endpoints

Reusing `count_rules_with_stats` from 4.1 for each endpoint.

In [ ]:
con = sqlite3.connect(db_path)

funnel_rows = []
for ep in ENDPOINTS:
    property_id = con.execute('select id from property_name where name = ?', (ep,)).fetchone()[0]
    n_indexed_with_property = con.execute(
        'select count(distinct compound_id) from compound_property where property_name_id = ?',
        (property_id,),
    ).fetchone()[0]
    funnel_rows.append({
        'endpoint': ep,
        'compounds with property': n_indexed_with_property,
        'rules with >=1 matched pair': count_rules_with_stats(con, property_id),
        'rules with >=5 pairs': count_rules_with_stats(con, property_id, min_pairs=5),
        '+ p<0.05': count_rules_with_stats(con, property_id, min_pairs=5, max_p_value=0.05),
    })

con.close()

funnel_df = pd.DataFrame(funnel_rows).set_index('endpoint')
funnel_df

## 5 — Representative rules per endpoint

Filter: ≥5 matched pairs, paired-t-test p<0.05, most specific environment radius ≤3 — exactly the paper's stated criteria (§5.4).

All six endpoints are shown. Note the paper's own MMP analysis (Figure 8) covers only **HLM, MDR1, and solubility** — the three it flagged as critical optimization endpoints. RLM and PPB_H/PPB_R are included here as an extension the paper did *not* perform, made possible by the ChEMBL-augmented labels in `df_sdf`; they are not recreations of any published result. The paper comparison in §7 is restricted to HLM/MDR1/SOL.

In [ ]:
rules_by_endpoint = {}
for ep in ENDPOINTS:
    rules = significant_rules(db_path, property_name=ep, max_radius=3, min_pairs=5, max_p_value=0.05)
    rules_by_endpoint[ep] = rules
    print(f'{ep}: {len(rules)} significant rules')

In [ ]:
for ep, rules in rules_by_endpoint.items():
    print(f'--- {ep} ---')
    display(rules[['from_smiles', 'to_smiles', 'n_pairs', 'mean_change', 'std_change', 'p_value']])

## 6 — Summary: significant rules per endpoint

In [ ]:
counts = pd.Series({ep: len(rules) for ep, rules in rules_by_endpoint.items()})

fig, ax = plt.subplots(figsize=(5, 3.5))
counts.plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('significant rules (>=5 pairs, p<0.05, radius<=3)')
ax.set_xlabel('endpoint')
ax.set_title('MMP rules surviving significance filter, per endpoint')
plt.tight_layout()
plt.savefig('../figures/section5_mmp_significant_rules_per_endpoint.png', dpi=150)
plt.show()

## 7 — Comparison to the paper's Figure 8

The paper's Figure 8 lists 38 representative transforms with their measured ΔHLM / ΔMDR1 / ΔSolubility (mean ± std, and nPairs) from the >25,000-compound internal database. Every transform shown there already passed the paper's ≥5-pairs-and-p<0.05 filter (that's the criterion for appearing in Figure 8), so the paper column below is significant by construction. Here we check the *same named transforms* against our own database — do they occur, with how many pairs, and does the direction/magnitude of the shift agree?

We restrict to the paper's **group-1 small-functional-group transforms (1–6)** — H↔CH₃, H↔F, H↔OH, H↔Cl, H↔NH₂, CH₃↔C≡N. These are the highest-pair-count transforms in the paper (up to 1243 pairs) and the ones we can encode unambiguously as mmpdb rule SMILES. The ring-system transforms (8–38) need exact fragment SMILES with the right attachment-point position; a hand-written SMILES that doesn't canonicalize byte-identically to mmpdb's stored form would silently return "absent" even if the rule were present — so rather than risk false negatives, §7.1 below instead lists *everything* we actually have at ≥5 pairs and confirms none of 8–38 are among them.

**Two notes on how our numbers are computed:**
- **Environment radius.** mmpdb stores each rule at radii 0–5. Radius 0 is the aggregate over all local environments (the most pairs); higher radii split those pairs by surrounding context. The paper's large nPairs (e.g. 993 for transform 1) indicate radius-0 aggregates, so this comparison reads mmpdb at **radius 0**. This differs from §5, which reports each rule at its most *specific* qualifying environment — so the same transform can show different (n, mean, p) between §5 and §7. Transform 1 (H↔CH₃, HLM) is exactly this case: §5 shows it at radius 1 (n=6, p=0.025, which is why it passed §5's filter), while §7 shows the radius-0 aggregate (n=18, p=0.050).
- **Direction.** The paper reports each transform as H→X (adding the group); mmpdb stores rules in a fixed canonical order that may be the reverse. The lookup detects the stored direction and flips the sign of the mean so every number is in the paper's H→X convention.

In [ ]:
# Paper Figure 8, group 1 (small functional groups), transcribed in the paper's H->X direction.
# Each entry: (label, from_smiles, to_smiles, {endpoint: (paper_mean, paper_std, paper_n)}).
# Blank cells in Figure 8 (e.g. transform 5 solubility) are simply omitted from the dict.
PAPER_FIG8 = [
    ('1: H->CH3',  '[*:1][H]', '[*:1]C',   {'HLM': (0.14, 0.29, 993), 'MDR1': (0.05, 0.33, 837), 'SOL': (-0.12, 0.50, 372)}),
    ('2: H->F',    '[*:1][H]', '[*:1]F',   {'HLM': (0.05, 0.23, 1243), 'MDR1': (-0.06, 0.30, 1170), 'SOL': (-0.15, 0.52, 560)}),
    ('3: H->OH',   '[*:1][H]', '[*:1]O',   {'HLM': (-0.23, 0.35, 66), 'MDR1': (0.89, 0.52, 23), 'SOL': (0.11, 0.10, 9)}),
    ('4: H->Cl',   '[*:1][H]', '[*:1]Cl',  {'HLM': (0.17, 0.40, 287), 'MDR1': (-0.12, 0.35, 204), 'SOL': (-0.34, 0.53, 89)}),
    ('5: H->NH2',  '[*:1][H]', '[*:1]N',   {'HLM': (-0.24, 0.42, 62), 'MDR1': (0.28, 0.50, 31)}),
    ('6: CH3->CN', '[*:1]C',   '[*:1]C#N', {'HLM': (-0.22, 0.41, 51), 'MDR1': (0.38, 0.37, 42), 'SOL': (-0.32, 0.71, 25)}),
]
PAPER_ENDPOINTS = ['HLM', 'MDR1', 'SOL']


def lookup_transform(con, paper_from, paper_to, property_id):
    """Return (mean, n_pairs, p_value) for a transform in the paper's from->to direction, or None.

    Reads the radius-0 (whole-transform aggregate) statistics, to match the paper's large-nPairs
    reporting. Finds the rule regardless of mmpdb's stored orientation, and flips the mean's sign
    if mmpdb stored the reverse direction.
    """
    rule_row = con.execute(
        '''select s1.smiles, r.id
           from rule r
           join rule_smiles s1 on r.from_smiles_id = s1.id
           join rule_smiles s2 on r.to_smiles_id = s2.id
           where (s1.smiles = ? and s2.smiles = ?) or (s1.smiles = ? and s2.smiles = ?)''',
        (paper_from, paper_to, paper_to, paper_from),
    ).fetchone()
    if rule_row is None:
        return None
    stored_from, rule_id = rule_row

    stat_row = con.execute(
        '''select res.count, res.avg, res.p_value
           from rule_environment_statistics res
           join rule_environment re on res.rule_environment_id = re.id
           where re.rule_id = ? and res.property_name_id = ? and re.radius = 0''',
        (rule_id, property_id),
    ).fetchone()
    if stat_row is None:
        return None

    n_pairs, mean_change, p_value = stat_row
    if stored_from != paper_from:            # mmpdb stored the reverse direction
        mean_change = -mean_change
    return mean_change, n_pairs, p_value


con = sqlite3.connect(db_path)
property_ids = {ep: con.execute('select id from property_name where name = ?', (ep,)).fetchone()[0]
                for ep in PAPER_ENDPOINTS}

comparison_rows = []
for label, paper_from, paper_to, paper_vals in PAPER_FIG8:
    for ep in PAPER_ENDPOINTS:
        paper = paper_vals.get(ep)
        ours = lookup_transform(con, paper_from, paper_to, property_ids[ep])
        comparison_rows.append({
            'transform': label,
            'endpoint': ep,
            'paper mean': f'{paper[0]:+.2f}' if paper else '',
            'paper n': paper[2] if paper else '',
            'our mean': f'{ours[0]:+.2f}' if ours else '(absent)',
            'our n': ours[1] if ours else '',
            'our p': f'{ours[2]:.3f}' if ours else '',
        })

con.close()

fig8_comparison = pd.DataFrame(comparison_rows)
fig8_comparison

### 7.1 — What we actually have: every ≥5-pair transform for the paper's three endpoints

Rather than probe for each of the paper's 38 named transforms one by one (error-prone for the ring systems, per the note above), this lists the *complete* set of transforms in our database that clear the ≥5-pair bar at radius 0, for HLM/MDR1/SOL. This is the exhaustive answer to "which of the paper's transforms could we possibly recover" — anything not in these lists simply isn't in our data at usable pair counts.

(The `our p` column here is unfiltered — a rule appears if it has ≥5 pairs, regardless of significance. Compare against §5, which additionally requires p<0.05.)

In [ ]:
con = sqlite3.connect(db_path)

for ep in PAPER_ENDPOINTS:
    property_id = property_ids[ep]
    all_rules = pd.read_sql_query(
        '''select s1.smiles as from_smiles, s2.smiles as to_smiles,
                  res.count as n_pairs, round(res.avg, 3) as mean_change, round(res.p_value, 4) as p_value
           from rule_environment_statistics res
           join rule_environment re on res.rule_environment_id = re.id
           join rule r on re.rule_id = r.id
           join rule_smiles s1 on r.from_smiles_id = s1.id
           join rule_smiles s2 on r.to_smiles_id = s2.id
           where res.property_name_id = ? and re.radius = 0 and res.count >= 5
           order by res.count desc''',
        con, params=(property_id,),
    )
    print(f'=== {ep}: {len(all_rules)} transforms with >=5 pairs (radius 0) ===')
    display(all_rules)

con.close()

### 7.2 — Structural pairs vs assay-measured pairs: where the data actually runs out

The ≥5-pair bar counts pairs where **both** molecules have the endpoint measured. That is a much harsher filter than "does this transformation occur in our compounds at all." To see which scarcity is the real bottleneck, the table below reports, for a few representative transforms, the **structural pair count** (molecule pairs making that change, ignoring assay data) alongside the **assay-measured pair count** per endpoint.

The pattern: transformations the paper reports are often *structurally present* in our set — even ring swaps like piperidine→morpholine — but almost none of those pairs were run through the same assay on both sides, so they never reach ≥5 usable pairs. Structure is plentiful; the paired labels are what run out. (H→NH₂ is the one true structural absence — it doesn't occur at all.)

In [ ]:
# Representative transforms spanning the paper's groups: two common functional-group swaps,
# one that occurs structurally but is never co-assayed, one true structural absence, and
# three ring-system transforms from groups 2-3.
REPRESENTATIVE_TRANSFORMS = [
    ('1: H->CH3 (group 1, common)',      '[*:1][H]',      '[*:1]C'),
    ('2: H->F (group 1, common)',        '[*:1][H]',      '[*:1]F'),
    ('6: CH3->CN (group 1)',             '[*:1]C',        '[*:1]C#N'),
    ('5: H->NH2 (group 1)',              '[*:1][H]',      '[*:1]N'),
    ('8: gem-diMe->cyclopropane (ring)', '[*:1]C(C)C',    '[*:1]C1CC1'),
    ('9: iPr->cyclobutane (ring)',       '[*:1]C(C)C',    '[*:1]C1CCC1'),
    ('17: piperidine->morpholine (ring)','[*:1]N1CCCCC1', '[*:1]N1CCOCC1'),
]


def structural_and_assay_pairs(con, from_smiles, to_smiles):
    """Return (rule_found, total_structural_pairs, {endpoint: assay_measured_pairs})."""
    rule_row = con.execute(
        '''select r.id from rule r
           join rule_smiles s1 on r.from_smiles_id = s1.id
           join rule_smiles s2 on r.to_smiles_id = s2.id
           where (s1.smiles = ? and s2.smiles = ?) or (s1.smiles = ? and s2.smiles = ?)''',
        (from_smiles, to_smiles, to_smiles, from_smiles),
    ).fetchone()
    if rule_row is None:
        return False, 0, {ep: 0 for ep in PAPER_ENDPOINTS}
    rule_id = rule_row[0]

    structural = con.execute(
        '''select count(*) from pair
           where rule_environment_id in (select id from rule_environment where rule_id = ?)''',
        (rule_id,),
    ).fetchone()[0]

    assay = {}
    for ep in PAPER_ENDPOINTS:
        stat = con.execute(
            '''select res.count from rule_environment_statistics res
               join rule_environment re on res.rule_environment_id = re.id
               where re.rule_id = ? and res.property_name_id = ? and re.radius = 0''',
            (rule_id, property_ids[ep]),
        ).fetchone()
        assay[ep] = stat[0] if stat else 0
    return True, structural, assay


con = sqlite3.connect(db_path)
property_ids = {ep: con.execute('select id from property_name where name = ?', (ep,)).fetchone()[0]
                for ep in PAPER_ENDPOINTS}

scarcity_rows = []
for label, from_smiles, to_smiles in REPRESENTATIVE_TRANSFORMS:
    found, structural, assay = structural_and_assay_pairs(con, from_smiles, to_smiles)
    scarcity_rows.append({
        'transform': label,
        'structural pairs': structural if found else '(not in DB)',
        'HLM pairs': assay['HLM'],
        'MDR1 pairs': assay['MDR1'],
        'SOL pairs': assay['SOL'],
    })

con.close()

scarcity_df = pd.DataFrame(scarcity_rows)
scarcity_df

## 8 — Discussion

**The pipeline recreates cleanly.** Fragmentation and indexing use mmpdb's own defaults (confirmed to match the paper's stated parameters verbatim by reading the mmpdb source), and the significance filter (≥5 pairs, p<0.05, radius≤3) is a direct SQL reimplementation of what mmpdb computes internally. The `df_sdf` source (§2) matters: it matches the paper's standardization and recovers the ChEMBL-augmented labels the public CSV drops.

**Against the paper's Figure 8, the story is data quantity, not method.** For the paper's three endpoints (HLM, MDR1, solubility), our public-data recreation is drastically sparser: only 1 significant rule for HLM, 0 for MDR1 and solubility, versus the dozens in Figure 8. But §7 shows this is a sample-size effect, not a discrepancy in the chemistry — and the sample-size bottleneck is specifically the *paired assay labels*, not the structures. Of the paper's group-1 transforms, only **1 (H→CH₃) and 2 (H→F)** have ≥5 pairs with the endpoint measured on both molecules across all three endpoints; **4 (H→Cl)** reaches ≥5 only for MDR1; **3 (H→OH) and 6 (CH₃→C≡N)** occur structurally (54 and 30 matched pairs respectively) but *zero* of those pairs have both molecules assayed; and **5 (H→NH₂)** does not occur in our set at all.

**The ring-system transforms (8–38) exist structurally but are label-starved, not chemically absent.** Several are present as real matched pairs — gem-dimethyl→cyclopropane (24 structural pairs), piperidine→morpholine (18), isopropyl→cyclobutane (6) — yet have 0–2 pairs with HLM/MDR1/solubility measured on both members, so none clear the ≥5 bar. §7.1's exhaustive ≥5-pair list therefore contains only small functional-group swaps (C↔H, F↔H, Cl↔H, OC↔H, C↔F) plus one aromatic positional change: not because our compounds lack these ring changes, but because the public set rarely measured the same assay on both sides of a ring-swap pair. This is the §4.1 funnel again — structure is plentiful, paired labels are the scarce resource.

**Where we do have pairs, the agreement is good.** H→CH₃ shifts HLM by +0.15 in our data vs +0.14 in the paper, and it's the one HLM transform clearing p<0.05 (borderline, p=0.050). Where pairs are few, estimates are unreliable — H→CH₃ on MDR1 comes out the *opposite* sign to the paper (−0.14 vs +0.05) on 17 pairs at p=0.12, i.e. noise, not a real contradiction.

**PPB is an extension beyond the paper, not a recreation.** The paper never performed MMP on plasma protein binding — Figure 8 is HLM/MDR1/solubility only. Our run happens to find 21 significant PPB_H rules with interpretable SAR (F→Cl on anilines lowering binding; halogen/alkyl-chain trends consistent across related rules), but this reflects only that our ChEMBL-augmented dataset has dense PPB_H coverage on well-connected compounds — there is no published result to compare it to. It is included because the data supports it, and it doubles as a useful internal positive control: it shows the pipeline *does* recover rich, sensible rules when the pair counts are there, which is exactly what HLM/MDR1/solubility lack in this public set.

**Takeaway for this project.** MMP rule discovery is more data-hungry than the ML models studied elsewhere here — a single transform needs ≥5 compound *pairs* sharing both a structural neighbour and a measured label, which the ~2,900-compound public set rarely provides for the paper's endpoints. A natural follow-up (deferred) is to subsample and track how the significant-rule count degrades with N, using PPB_H — the one endpoint with real signal — as the case where a shrinking-N trend would actually be visible.